# 06. 전이학습 (Transfer Learning)

한 작업에서 학습한 모델의 지식을, 데이터가 적은 다른 작업에 재사용하는 방법입니다.

이 노트북에서는 FashionMNIST(28×28 흑백 의류 이미지, 10개 클래스)를 두 작업으로 나눕니다.
- **원본 작업(source)**: 클래스 0~4 (T-shirt, Trouser, Pullover, Dress, Coat)를 많은 데이터로 학습
- **목표 작업(target)**: 처음 보는 클래스 5~9 (Sandal, Shirt, Sneaker, Bag, Ankle boot)를 **클래스당 50장**만으로 학습

**학습 목표**
- 사전학습한 모델의 특징 추출부(backbone)를 다른 작업에 재사용한다
- 레이어를 고정(freeze)하고 분류층만 교체해 학습한다(feature extraction)
- 전체를 작은 학습률로 미세조정(fine-tuning)하고, 처음부터 학습한 경우와 비교한다

데이터: `data/downloads/` (FashionMNIST, 첫 실행 시 자동 다운로드, git 제외), 사전학습 모델: `data/pretrained_source.pt` (git 제외)

In [ ]:
import torch

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}, device: {device}")

## 1. 데이터 준비

클래스 번호로 원본/목표 작업을 나누고, 목표 작업의 라벨은 5~9를 0~4로 바꿉니다.

In [ ]:
from pathlib import Path
from torchvision import datasets
from torch.utils.data import DataLoader, TensorDataset

DATA = Path("data")
train_full = datasets.FashionMNIST(DATA / "downloads", train=True, download=True)
test_full = datasets.FashionMNIST(DATA / "downloads", train=False, download=True)
classes = train_full.classes

# 데이터가 작으므로 전체를 텐서로 올려 정규화 (평균 0.2860, 표준편차 0.3530)
def to_tensors(ds):
    x = (ds.data.float() / 255 - 0.2860) / 0.3530
    return x.unsqueeze(1), ds.targets

x_train, y_train = to_tensors(train_full)
x_test, y_test = to_tensors(test_full)

# 원본 작업: 클래스 0~4 전체
src = y_train < 5
source_train = TensorDataset(x_train[src], y_train[src])
src_test = y_test < 5
source_test = TensorDataset(x_test[src_test], y_test[src_test])

# 목표 작업: 클래스 5~9, 학습은 클래스당 50장만
N_PER_CLASS = 50
g = torch.Generator().manual_seed(0)
idx = torch.cat([
    (y_train == c).nonzero().flatten()[torch.randperm((y_train == c).sum().item(), generator=g)[:N_PER_CLASS]]
    for c in range(5, 10)
])
target_train = TensorDataset(x_train[idx], y_train[idx] - 5)
tgt_test = y_test >= 5
target_test = TensorDataset(x_test[tgt_test], y_test[tgt_test] - 5)

print(f"원본 작업: train {len(source_train)} | test {len(source_test)} | {classes[:5]}")
print(f"목표 작업: train {len(target_train)} | test {len(target_test)} | {classes[5:]}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for c in range(10):
    ax = axes[c // 5, c % 5]
    ax.imshow(x_train[(y_train == c).nonzero()[0, 0], 0], cmap="gray")
    ax.set_title(f"{c}: {classes[c]}", fontsize=9)
    ax.axis("off")
axes[0, 0].set_ylabel("source")
plt.suptitle("top: source classes (0-4) / bottom: target classes (5-9)")
plt.tight_layout()
plt.show()

## 2. 모델

특징 추출부(`features`)와 분류층(`head`)을 분리해 두면 분류층만 바꿔 끼우기 쉽습니다.

In [ ]:
from torch import nn

class CNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.head = nn.Linear(128, num_classes)

    def forward(self, x):
        return self.head(self.features(x))

def train(model, dataset, epochs, lr, params=None, frozen_features=False, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(params if params is not None else model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        if frozen_features:
            model.features.eval()   # 고정한 특징추출부는 BatchNorm 통계도 갱신하지 않음
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()

@torch.no_grad()
def accuracy(model, dataset):
    model.eval()
    correct = 0
    for x, y in DataLoader(dataset, batch_size=1000):
        correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
    return correct / len(dataset)

## 3. 원본 작업으로 사전학습

클래스 0~4의 이미지 30,000장으로 학습하고, 가중치를 `data/`에 저장합니다.

In [ ]:
pretrained_path = DATA / "pretrained_source.pt"

source_model = CNN().to(device)
train(source_model, source_train, epochs=2, lr=1e-3)
print(f"원본 작업 test acc: {accuracy(source_model, source_test):.3f}")
torch.save(source_model.state_dict(), pretrained_path)
print("저장:", pretrained_path)

## 4. 목표 작업: 세 가지 방법 비교

| 방법 | 초기 가중치 | 학습하는 파라미터 |
|---|---|---|
| A. 처음부터 학습 | 무작위 | 전체 |
| B. 특징 추출 (feature extraction) | 사전학습 | 새 분류층만 (`features` 고정) |
| C. 미세조정 (fine-tuning) | 사전학습 | 전체 (작은 학습률) |

목표 작업 데이터가 250장뿐이므로 같은 epoch 수로 학습해 비교합니다.

In [ ]:
EPOCHS = 30

def load_pretrained():
    model = CNN().to(device)
    model.load_state_dict(torch.load(pretrained_path, map_location=device))
    model.head = nn.Linear(128, 5).to(device)   # 분류층을 새 작업용으로 교체
    return model

results = {}

# A. 처음부터 학습
torch.manual_seed(0)
scratch = CNN().to(device)
train(scratch, target_train, EPOCHS, lr=1e-3)
results["A. scratch"] = accuracy(scratch, target_test)

# B. 특징 추출: features 고정(BatchNorm 통계 포함), head만 학습
torch.manual_seed(0)
frozen = load_pretrained()
for p in frozen.features.parameters():
    p.requires_grad = False
trainable = sum(p.numel() for p in frozen.parameters() if p.requires_grad)
print(f"B. 학습 파라미터 {trainable:,} / 전체 {sum(p.numel() for p in frozen.parameters()):,}")
train(frozen, target_train, EPOCHS, lr=1e-3, params=frozen.head.parameters(), frozen_features=True)
results["B. feature extraction"] = accuracy(frozen, target_test)

# C. 미세조정: 전체를 작은 학습률로
torch.manual_seed(0)
finetuned = load_pretrained()
train(finetuned, target_train, EPOCHS, lr=1e-4)
results["C. fine-tuning"] = accuracy(finetuned, target_test)

for k, v in results.items():
    print(f"{k:24s} test acc {v:.3f}")

In [ ]:
plt.figure(figsize=(7, 3))
names, accs = list(results), list(results.values())
bars = plt.barh(names, accs, color=["gray", "tab:blue", "tab:green"])
plt.bar_label(bars, fmt="%.3f", padding=3)
plt.xlim(0, 1)
plt.gca().invert_yaxis()
plt.title(f"Target task accuracy ({N_PER_CLASS} images per class)")
plt.xlabel("test accuracy")
plt.tight_layout()
plt.show()

## 5. 예측 확인 (미세조정 모델)

In [ ]:
x, y = next(iter(DataLoader(target_test, batch_size=16, shuffle=True, generator=torch.Generator().manual_seed(1))))
finetuned.eval()
with torch.no_grad():
    pred = finetuned(x.to(device)).argmax(1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, i in zip(axes.flat, range(16)):
    ax.imshow(x[i, 0], cmap="gray")
    ax.set_title(classes[pred[i] + 5], color="green" if pred[i] == y[i] else "red", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. 실무에서: ImageNet 사전학습 모델 사용

실제로는 대규모 데이터셋(ImageNet)으로 사전학습된 모델을 내려받아 같은 방식으로 사용합니다. 방법은 이 노트북과 같습니다: 분류층을 교체하고, backbone을 고정하거나 미세조정합니다. (가중치를 `download.pytorch.org`에서 받으므로 인터넷이 필요합니다.)

```python
from torchvision import models

weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights)
for p in model.parameters():                       # backbone 고정
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, num_classes)  # 분류층 교체
preprocess = weights.transforms()                  # 사전학습 때와 같은 전처리 (224×224, ImageNet 정규화)
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
```

## 요약
- 사전학습 모델의 특징 추출부는 처음 보는 클래스에도 쓸 수 있는 일반적인 특징을 담고 있다
- 데이터가 적을수록 전이학습의 효과가 크다
- 특징 추출(freeze)은 빠르고 과적합에 강하며, 미세조정은 작은 학습률로 전체를 조정해 보통 더 높은 성능을 낸다
- BatchNorm이 있는 모델을 고정할 때는 `eval()` 상태도 함께 고정해야 한다